# User Behavior-based Auto Engagement Chatbot — Colab Demo

Notebook นี้รันตาม Flow ตั้งแต่ต้นจนจบ: Dataset → Preprocessing → Train Model → Behavior Engine → Decision Engine → API → ทดสอบสำหรับ Make/Facebook

In [ ]:
# STEP 0: แตกไฟล์ ZIP แล้วเข้าโฟลเดอร์โปรเจกต์
# ถ้า Upload zip มาใน Colab แล้ว ให้รัน Cell นี้
!unzip -o colab_auto_engagement_make_project_ready.zip
%cd colab_auto_engagement_make_project

In [ ]:
# STEP 1: ติดตั้ง Library
!pip install -q -r requirements.txt

In [ ]:
# STEP 2: เตรียม Dataset + Clean Text
!python 01_prepare_dataset.py

In [ ]:
# STEP 3: Train Sentiment Model และ Category Model
!python 02_train_models.py

In [ ]:
# STEP 4: สร้าง User Behavior Features
!python 03_build_behavior.py

In [ ]:
# STEP 5: ทดสอบข้อความแบบไม่ต้องเปิด API
!python 04_test_prediction.py

In [ ]:
# STEP 6: ดู Metrics ของโมเดล
import json
with open("outputs/model_metrics.json", "r", encoding="utf-8") as f:
    metrics = json.load(f)
print("Sentiment Accuracy:", metrics["sentiment_accuracy"])
print("Category Accuracy:", metrics["category_accuracy"])

In [ ]:
# STEP 7: เปิด API Server ใน Colab
# หมายเหตุ: Cell นี้จะรันค้างไว้ ให้เปิด Runtime อีก cell หรือหยุดเมื่อทดสอบเสร็จ
!uvicorn app.main:app --host 0.0.0.0 --port 8000

## ทดสอบ API ใน Cell ใหม่
รัน Cell ด้านล่างหลังจาก API Server เปิดอยู่

In [ ]:
import requests
payload = {
    "user_id": "facebook_user_001",
    "message": "อาหารช้ามาก ไม่ประทับใจเลย",
    "channel": "facebook",
    "display_name": "Customer Test",
    "source": "make"
}
res = requests.post("http://127.0.0.1:8000/make/predict", json=payload)
res.json()

## สร้าง Public URL เพื่อเอาไปใส่ Make
ต้องมี ngrok token ก่อน จากนั้นรัน Cell นี้ แล้วนำ URL ที่ได้ไปใช้ใน Make: `https://xxxx.ngrok-free.app/make/predict`

In [ ]:
from pyngrok import ngrok
# ใส่ ngrok token ของตัวเอง
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")
public_url = ngrok.connect(8000)
print("Public URL:", public_url)
print("Make URL:", str(public_url) + "/make/predict")

In [ ]:
# STEP 8: สร้างตารางผลสำหรับบทความ
!python 06_create_research_tables.py

In [ ]:
# STEP 9: ดาวน์โหลด outputs ทั้งหมด
!zip -r outputs_for_article.zip outputs models
from google.colab import files
files.download("outputs_for_article.zip")